# Uloha 1 - Identita
Neuronova siet 4-2-4 s aktivacnou funkciou Sigmoid.

**Autor:** Ostapchuk

In [ ]:
# zakladne kniznice
import torch
import torch.nn as nn
import numpy as np
import os

# seed pre opakovatelnost vysledkov
torch.manual_seed(42)
np.random.seed(42)

# priecinok na ukladanie modelov
os.makedirs('models', exist_ok=True)

print("PyTorch verzia:", torch.__version__)

In [ ]:
# Neuronova siet: 4 vstupy -> 2 skryte -> 4 vystupy
# Skryta vrstva ma len 2 neurony, takze siet musi skomprimovat informaciu
# Sigmoid aktivacia dava hodnoty medzi 0 a 1, co je dobre pre binarne data

class IdentityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(4, 2)   # kompresia 4 -> 2
        self.output = nn.Linear(2, 4)   # dekodovanie 2 -> 4
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

print("Siet vytvorena: 4-2-4, Sigmoid")

In [ ]:
# Trenovacia funkcia
# Kazdu epochu sa nahodne premiesa poradie vektorov
# Pre kazdy vektor sa pocita SSE chyba a upravia vahy

def train_model(model, data, epochs, lr, print_every=100, start_epoch=1):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    posledna_chyba = 0.0
    for epoch in range(start_epoch, start_epoch + epochs):
        indices = torch.randperm(data.size(0))  # nahodne poradie
        global_error = 0.0

        for i in indices:
            x = data[i]
            output = model(x)
            loss = torch.sum((output - x) ** 2)  # SSE
            global_error += loss.item()

            optimizer.zero_grad()   # vynulovanie gradientov
            loss.backward()         # spatny prechod
            optimizer.step()        # aktualizacia vah

        posledna_chyba = global_error

        if epoch == start_epoch or epoch % print_every == 0 or epoch == start_epoch + epochs - 1:
            print(f"Epoch {epoch:6d}   Global error {global_error:.5f}")

    return posledna_chyba

print("Trenovacia funkcia pripravena")

In [ ]:
# Testovanie modelu
# Accuracy = kolko bitov je spravnych po zaokruhleni
# Reliability = kolko vystupov je blizko k 0 alebo 1 (v ramci epsilon)

def test_model(model, data, epsilon=0.2):
    model.eval()

    print("\nTestovanie")
    print(f"{'Input':<12} {'Output':<12} {'Response':<28} {'Error':>7} {'Acc':>6} {'Rel':>6}")
    print("-" * 75)

    celk_acc = 0
    celk_rel = 0
    spravne_vektory = 0
    pocet = data.size(0)

    with torch.no_grad():
        for x in data:
            response = model(x)
            rounded = torch.round(response)

            error = torch.sum((response - x) ** 2).item()

            correct_bits = (rounded == x).sum().item()
            accuracy = correct_bits / len(x) * 100
            celk_acc += accuracy

            # reliability - ci su hodnoty blizko 0 alebo 1
            reliable = ((response <= epsilon) | (response >= 1 - epsilon)).sum().item()
            reliability = reliable / len(x) * 100
            celk_rel += reliability

            if (rounded == x).all():
                spravne_vektory += 1

            inp = ' '.join([f'{int(v)}' for v in x])
            out = ' '.join([f'{int(v)}' for v in rounded])
            resp = ' '.join([f'{v:.2f}' for v in response])
            print(f"{inp:<12} {out:<12} {resp:<28} {error:>7.3f} {accuracy:>5.0f}% {reliability:>5.0f}%")

    avg_acc = celk_acc / pocet
    avg_rel = celk_rel / pocet
    print(f"\nSpravne vektory: {spravne_vektory}/{pocet}")
    print(f"Priemerna accuracy:    {avg_acc:.1f}%")
    print(f"Priemerna reliability: {avg_rel:.1f}%")

    model.train()
    return spravne_vektory, avg_acc, avg_rel

print("Testovacia funkcia pripravena")

In [ ]:
# Pomocna funkcia - spusti cely experiment (vytvorenie siete, trening, test, ulozenie)

def run_experiment(name, data, lr_schedule, epsilon=0.2, print_every=100):
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {name}")
    print(f"{'='*60}")

    torch.manual_seed(42)
    model = IdentityNet()

    total_epochs = sum(e for e, _ in lr_schedule)
    print(f"Pocet epoch: {total_epochs}, lr schedule: {lr_schedule}")

    # trenovanie - moze mat viac faz s roznym lr
    start = 1
    for epochs, lr in lr_schedule:
        if len(lr_schedule) > 1:
            print(f"\n--- lr={lr}, {epochs} epoch ---")
        train_model(model, data, epochs, lr, print_every=print_every, start_epoch=start)
        start += epochs

    spravne, acc, rel = test_model(model, data, epsilon=epsilon)

    filename = f"models/{name.replace(' ', '_').lower()}.pth"
    torch.save(model.state_dict(), filename)
    print(f"Model ulozeny: {filename}")

    return model, spravne, acc, rel

print("Funkcia run_experiment pripravena")

---
# Poduloha 1: Identita pre 5 vektorov

Siet sa ma naucit reprodukovat 5 vybranych 4-bitovych vektorov. Ocakavame 100% uspesnost.

In [ ]:
# 5 binarnych vektorov - vstup a vystup su rovnake (funkcia identity)
data_5 = torch.tensor([
    [1, 1, 0, 0],
    [0, 0, 1, 1],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 0, 0]
], dtype=torch.float32)

print("Data pre podulohu 1:")
print(data_5)

In [ ]:
# Experiment 1.1 - zakladny pokus, lr=0.5, 1000 epoch
model_1_1, s_1_1, a_1_1, r_1_1 = run_experiment(
    name="Poduloha1_Exp1",
    data=data_5,
    lr_schedule=[(1000, 0.5)],
    print_every=200
)

### Experiment 1.1 (baseline)
Zakladny pokus s lr=0.5 a 1000 epochami. Sluzi na porovnanie s dalsimi experimentmi.
Lr 0.5 je rozumna stredna hodnota - nie prilis mala, nie prilis velka.

In [ ]:
# Experiment 1.2 - vyssie lr=2.0, viac epoch
model_1_2, s_1_2, a_1_2, r_1_2 = run_experiment(
    name="Poduloha1_Exp2",
    data=data_5,
    lr_schedule=[(3000, 2.0)],
    print_every=500
)

### Experiment 1.2 (vysoke lr)
Lr=2.0 a 3000 epoch. Vyssie lr znamena vacsie kroky pri uceni, co moze pomoct
najst riesenie rychlejsie, ale tiez to moze byt nestabilne.

In [ ]:
# Experiment 1.3 - krokovy lr: zaciname vysoko a postupne znizujeme
# -- Pozn: napad s krokovym lr som nasiel s pomocou AI, je to bezna technika
#    kde sa zacina s velkym lr a postupne sa znizuje pre jemnejsie doladenie --
model_1_3, s_1_3, a_1_3, r_1_3 = run_experiment(
    name="Poduloha1_Exp3",
    data=data_5,
    lr_schedule=[(1000, 2.0), (1000, 0.5), (500, 0.05)],
    print_every=500
)

### Experiment 1.3 (krokovy lr)
Lr zacina na 2.0, potom 0.5, a nakoniec 0.05. Vysoke lr na zaciatku rychlo priblizi
siet k rieseniu a nizke lr na konci ju presnejsie doladi. Toto by malo dat najlepsiu reliability.

In [ ]:
# Porovnanie experimentov podulohy 1
print("=" * 55)
print("POROVNANIE - PODULOHA 1 (5 vektorov)")
print("=" * 55)
print(f"{'Experiment':<22} {'Spravne':>8} {'Accuracy':>10} {'Reliability':>12}")
print("-" * 55)
print(f"{'Exp 1.1 (lr=0.5)':<22} {s_1_1:>5}/5   {a_1_1:>8.1f}%  {r_1_1:>10.1f}%")
print(f"{'Exp 1.2 (lr=2.0)':<22} {s_1_2:>5}/5   {a_1_2:>8.1f}%  {r_1_2:>10.1f}%")
print(f"{'Exp 1.3 (step lr)':<22} {s_1_3:>5}/5   {a_1_3:>8.1f}%  {r_1_3:>10.1f}%")

---
# Poduloha 2: Identita pre 16 vektorov

Siet sa ma naucit vsetkych 16 moznych 4-bitovych vektorov.
Je to tazsie, lebo 2 skryte neurony musia zakodovat 16 roznych vzorov.
Ciel je minimalne 9-10 spravnych vektorov.

In [ ]:
# Vsetky 4-bitove binarne vektory (2^4 = 16)
# -- Pozn: bitovy posun (>> a &) som si nasiel cez AI,
#    je to sposob ako z cisla dostat jednotlive bity,
#    napr. cislo 5 = 0101 -> [0, 1, 0, 1] --
data_16_list = []
for i in range(16):
    vector = [(i >> 3) & 1, (i >> 2) & 1, (i >> 1) & 1, i & 1]
    data_16_list.append(vector)

data_16 = torch.tensor(data_16_list, dtype=torch.float32)

print("Data pre podulohu 2 (16 vektorov):")
for idx, v in enumerate(data_16):
    print(f"  {idx:2d}: [{int(v[0])} {int(v[1])} {int(v[2])} {int(v[3])}]")

In [ ]:
# Experiment 2.1 - zakladny pokus, lr=0.5, 5000 epoch
model_2_1, s_2_1, a_2_1, r_2_1 = run_experiment(
    name="Poduloha2_Exp1",
    data=data_16,
    lr_schedule=[(5000, 0.5)],
    print_every=1000
)

### Experiment 2.1 (baseline)
Zakladne nastavenie pre 16 vektorov. S lr=0.5 a 5000 epochami.
16 vektorov je ovela tazsie nez 5, takze treba viac epoch.

In [ ]:
# Experiment 2.2 - vysoke lr=2.0, 10000 epoch
model_2_2, s_2_2, a_2_2, r_2_2 = run_experiment(
    name="Poduloha2_Exp2",
    data=data_16,
    lr_schedule=[(10000, 2.0)],
    print_every=2000
)

### Experiment 2.2 (vysoke lr)
Lr=2.0 a 10000 epoch. Vysoke lr pomoze prekonat lokalne minima,
ale moze sposobit ze siet bude "preskakovany" okolo optima.

In [ ]:
# Experiment 2.3 - krokovy lr, rovnaky princip ako v podulohe 1
# -- Pozn: tuto strategiu som uz pouzil v exp 1.3, tu len prisposobene
#    pre 16 vektorov s viac epochami --
model_2_3, s_2_3, a_2_3, r_2_3 = run_experiment(
    name="Poduloha2_Exp3",
    data=data_16,
    lr_schedule=[(5000, 2.0), (3000, 0.5), (2000, 0.05)],
    print_every=2000
)

### Experiment 2.3 (krokovy lr)
Rovnaka strategia ako exp 1.3, ale s viac epochami pre 16 vektorov.
Vysoke lr na zaciatku, potom stredne, a na konci nizke pre presne doladenie.

In [ ]:
# Porovnanie experimentov podulohy 2
print("=" * 55)
print("POROVNANIE - PODULOHA 2 (16 vektorov)")
print("=" * 55)
print(f"{'Experiment':<22} {'Spravne':>8} {'Accuracy':>10} {'Reliability':>12}")
print("-" * 55)
print(f"{'Exp 2.1 (lr=0.5)':<22} {s_2_1:>5}/16  {a_2_1:>8.1f}%  {r_2_1:>10.1f}%")
print(f"{'Exp 2.2 (lr=2.0)':<22} {s_2_2:>5}/16  {a_2_2:>8.1f}%  {r_2_2:>10.1f}%")
print(f"{'Exp 2.3 (step lr)':<22} {s_2_3:>5}/16  {a_2_3:>8.1f}%  {r_2_3:>10.1f}%")

---
# Parametre experimentov

| Parameter | Hodnota |
|-----------|---------|
| Topologia | 4 - 2 - 4 |
| Aktivacia | Sigmoid |
| Optimizer | SGD |
| Chyba | SSE |
| Epsilon | 0.2 |

### Poduloha 1

| Exp | LR | Epochy |
|-----|-----|--------|
| 1.1 | 0.5 | 1000 |
| 1.2 | 2.0 | 3000 |
| 1.3 | 2.0 → 0.5 → 0.05 | 2500 |

### Poduloha 2

| Exp | LR | Epochy |
|-----|-----|--------|
| 2.1 | 0.5 | 5000 |
| 2.2 | 2.0 | 10000 |
| 2.3 | 2.0 → 0.5 → 0.05 | 10000 |

---
# Zhrnutie

## Poduloha 1 (5 vektorov)
- 5 vektorov je relativne lahka uloha, siet zvladne 100% accuracy
- Krokovy lr (Exp 1.3) ma najlepsiu reliability
- Aj zakladny experiment s lr=0.5 funguje dobre

## Poduloha 2 (16 vektorov)
- 16 vektorov je ovela tazsie - 2 neurony musia zakodovat 16 vzorov
- Krokovy lr (Exp 2.3) dava najlepsie vysledky
- 100% accuracy nie je zarucena, ale aspon 9-10 spravnych je realne

## Co som zistil
1. Learning rate je dolezity - maly = pomale ucenie, velky = nestabilita
2. Krokovy lr (velky na zaciatku, maly na konci) funguje najlepsie
3. Nahodne poradie vektorov v kazdej epoche pomaha uceniu